# Experiment: Q-ERA Stage 6A Scalability Study

Run or inspect each Stage 6A gate independently while preserving the frozen v1.1 evidence. The core question is how formulation size, QUBO coupling, Classiq resources, and QAOA sample quality change for D = 4, 5, 6, and 8 with K = S = 3 and p = 1.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

cwd = Path.cwd().resolve()
if (cwd / 'stage6a').is_dir():
    IMPLEMENTATION_ROOT = cwd
elif (cwd.parent / 'stage6a').is_dir():
    IMPLEMENTATION_ROOT = cwd.parent
else:
    raise RuntimeError('Start this notebook from implementation/ or implementation/notebooks/.')
STAGE6A_ROOT = IMPLEMENTATION_ROOT / 'stage6a'
TABLES = STAGE6A_ROOT / 'artifacts' / 'scaling' / 'tables'
FIGURES = STAGE6A_ROOT / 'artifacts' / 'scaling' / 'figures'

def run_script(name: str, *args: object) -> None:
    command = [sys.executable, str(STAGE6A_ROOT / 'scripts' / name), *map(str, args)]
    subprocess.run(command, cwd=IMPLEMENTATION_ROOT, check=True)

print('Python:', sys.version.split()[0])
print('Implementation root:', IMPLEMENTATION_ROOT)


## Run policy

Local validation and evidence refreshes are safe to repeat. Classiq synthesis and execution upload the generated model/program to Classiq and are disabled by default. New external runs use unique labels so existing evidence is never overwritten.


In [ ]:
RUN_AUDIT = True
REBUILD_LOCAL_INPUTS = False
RUN_CLASSIQ_SYNTHESIS = False
RUN_CLASSIQ_QAOA = False
REFRESH_EVIDENCE = True
SYNTHESIS_TAG = 'notebook-01'
QAOA_RUN_TAG = 'notebook-01'


## Step A — audit frozen v1.1

Success means the original tests, artifact hashes, adaptive updates, and saved quantum reprocessing still agree.


In [ ]:
if RUN_AUDIT:
    run_script('audit_frozen_v11.py')
audit = json.loads((TABLES / 'adaptive_weight_audit.json').read_text(encoding='utf-8'))
audit


## Steps B–D — instances, acceptance, classical truth, and QUBOs

Enable `REBUILD_LOCAL_INPUTS` to deterministically regenerate the Stage 6A instance and benchmark artifacts. This does not call Classiq.


In [ ]:
if REBUILD_LOCAL_INPUTS:
    run_script('generate_instances.py')
    run_script('run_classical_qubo_scaling.py')
    run_script('create_scaling_qmod.py')
display(pd.read_csv(TABLES / 'scaling_classical.csv'))
display(pd.read_csv(TABLES / 'scaling_qubo.csv'))


## Step E — optional fresh Classiq synthesis

Set `RUN_CLASSIQ_SYNTHESIS = True` only when authenticated and online. Change `SYNTHESIS_TAG` for every new attempt. The canonical saved D=5, D=6, and D=8 syntheses are already present.


In [ ]:
if RUN_CLASSIQ_SYNTHESIS:
    for D in (5, 6, 8):
        run_script('synthesize_scaling_qaoa.py', '--D', D, '--artifact-tag', SYNTHESIS_TAG)
display(pd.read_csv(TABLES / 'scaling_synthesis.csv'))


## Step F — optional fresh D=5 and D=6 QAOA runs

Execution uses the canonical, hash-bound syntheses, 10 optimizer iterations × 512 shots, and 4,096 final shots. Change `QAOA_RUN_TAG` for every new attempt. D=8 is intentionally synthesis-only.


In [ ]:
if RUN_CLASSIQ_QAOA:
    for D in (5, 6):
        instance_id = f'qera-d{D}-seed{6100 + D}'
        run_name = f'{instance_id}-p1-{QAOA_RUN_TAG}'
        circuit_args = []
        if RUN_CLASSIQ_SYNTHESIS:
            circuit_stem = f'{instance_id}-p1-{SYNTHESIS_TAG}'
            circuit_dir = STAGE6A_ROOT / 'artifacts' / 'scaling' / 'circuits'
            circuit_args = ['--qprog', circuit_dir / f'{circuit_stem}.qprog', '--synthesis-manifest', circuit_dir / f'{circuit_stem}.synthesis.json']
        run_script('execute_scaling_qaoa.py', '--D', D, '--run-name', run_name, *circuit_args)
        run_script('process_scaling_qaoa.py', '--D', D, '--run-name', run_name)


## Step G — rebuild tables, figures, and findings from saved records


In [ ]:
if REFRESH_EVIDENCE:
    run_script('collect_synthesis_scaling.py')
    run_script('collect_scaling_evidence.py')
    run_script('generate_scaling_plots.py')
    run_script('write_scalability_findings.py')
qaoa = pd.read_csv(TABLES / 'scaling_qaoa.csv')
display(qaoa[['D', 'method', 'one_hot_probability', 'joint_feasible_probability', 'selected_relative_gap']])


In [ ]:
for figure in sorted(FIGURES.glob('*.png')):
    display(Markdown(f'### {figure.stem}'))
    display(Image(filename=str(figure)))
display(Markdown((STAGE6A_ROOT / 'SCALABILITY_FINDINGS.md').read_text(encoding='utf-8')))
